# Contract Clause Risk Analysis — QLoRA Fine-tuning
**Model:** LLaMA 3.1 8B Instruct  
**Method:** QLoRA (4-bit quantization + LoRA adapters)  
**Task:** Role-conditioned risk classification (Licensor / Licensee → Low / Medium / High)

### Before running:
1. **Accelerator** → Settings → Accelerator → **GPU T4 x2** (or P100)
2. **HF Token** → Add a Kaggle Secret named `HF_TOKEN` with your Hugging Face token (Notebook → Add-ons → Secrets). Request LLaMA 3.1 access at hf.co/meta-llama if needed.
3. **Dataset files** → Upload `train.jsonl` and `val.jsonl` as a Kaggle Dataset (+ Add data → New Dataset), then attach it. They will appear under `/kaggle/input/<dataset-name>/`.
   - Update `TRAIN_FILE` / `VAL_FILE` in the Config cell to match your dataset path.
4. **Internet** → Must be **ON** (Settings → Internet → On) to pull model weights from HF.
5. Outputs are saved to `/kaggle/working/qlora_output/` — download them from the Output tab when done.

## 1. Install Dependencies
Kaggle kernels ship with PyTorch pre-installed — we only need the QLoRA/LoRA packages.
Run this cell once and wait for it to finish before continuing.

In [ ]:
# %%capture
# 1. Install the base dependencies and your required transformers version normally
!pip install -q \
    numpy==1.26.4 \
    transformers<4.46.0 \
    triton==2.2.0 \
    bitsandbytes>=0.46.1 \
    accelerate==0.33.0 \
    datasets==2.21.0 \
    huggingface-hub==0.23.4 \
    scikit-learn \
    seaborn \
    matplotlib
# 2. Force-install peft and trl while ignoring their dependency demands
!pip install -q --no-deps peft==0.12.0 trl==0.8.6

print('✓ Packages force-installed')

## 2. Hugging Face Login
Reads your HF token from **Kaggle Secrets** (key: `HF_TOKEN`). Never hard-code tokens in notebooks.

In [ ]:
pip install -U bitsandbytes>=0.46.1

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

secrets = UserSecretsClient()
hf_token = secrets.get_secret('HF_TOKEN')
login(token=hf_token)
print('✓ Logged in to Hugging Face')

In [ ]:
!pip install "transformers<4.46.0"

In [ ]:
# pip install tyro>=0.5.11
!pip install trl==0.8.6

## 3. Config — all hyperparameters in one place

In [1]:
import os

# ── Paths ──────────────────────────────────────────────────────
# Update these to match your Kaggle Dataset path
TRAIN_FILE  = "/kaggle/input/datasets/udaysodhi/nlp-data/train_balanced.jsonl"  # ← edit dataset name
VAL_FILE    = "/kaggle/input/datasets/udaysodhi/nlp-data/val.jsonl"    # ← edit dataset name
OUTPUT_DIR  = "/kaggle/working/qlora_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Model ──────────────────────────────────────────────────────
MODEL_ID       = "meta-llama/Meta-Llama-3.1-8B-Instruct"

# ── QLoRA ──────────────────────────────────────────────────────
LORA_RANK      = 16    # size of the adapter matrices; higher = more capacity
LORA_ALPHA     = 32    # scaling factor; rule of thumb: 2x rank
LORA_DROPOUT   = 0.05
LORA_MODULES   = ["q_proj", "k_proj", "v_proj", "o_proj"]

# ── Training ───────────────────────────────────────────────────
EPOCHS         = 5
BATCH_SIZE     = 2     # per device; keep low for T4
GRAD_ACCUM     = 8     # effective batch = BATCH_SIZE * GRAD_ACCUM = 16
LR             = 2e-4
MAX_SEQ_LEN    = 512
WARMUP_RATIO   = 0.05

SEED           = 42

print(f'✓ Config set. Outputs → {OUTPUT_DIR}')
print(f'  Train: {TRAIN_FILE}')
print(f'  Val:   {VAL_FILE}')

✓ Config set. Outputs → /kaggle/working/qlora_output
  Train: /kaggle/input/datasets/udaysodhi/nlp-data/train_balanced.jsonl
  Val:   /kaggle/input/datasets/udaysodhi/nlp-data/val.jsonl


## 4. Load Dataset

In [2]:
from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files={"train": TRAIN_FILE, "validation": VAL_FILE}
)

print(dataset)
print("\nSample training example:")
print(dataset["train"][0]["text"])

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', '_meta'],
        num_rows: 3975
    })
    validation: Dataset({
        features: ['text', '_meta'],
        num_rows: 826
    })
})

Sample training example:
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are a legal risk analyst specializing in licensing agreements. Your task is to analyze a contract clause and assess its risk level from a specific party's perspective. Focus on the practical consequences for the party — what they stand to lose, what obligations they must perform, and how protected they are if things go wrong. As a rough guide: low risk implies little to no liability for the party, and high risk implies exposure.<|eot_id|><|start_header_id|>user<|end_header_id|>

Analyze the following contract clause from the perspective of the **Licensor**.

Clause:
"""
2.3 Notwithstanding anything to the contrary in this Agreement, PFHOF shall have the right to approve (in its sole and absolute discret

## 5. Load Model in 4-bit (QLoRA)

In [3]:
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)

# 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",              # NormalFloat4 — best for normally distributed weights
    bnb_4bit_compute_dtype=torch.bfloat16,  # compute in bf16 for stability
    bnb_4bit_use_double_quant=True,         # quantize quantization constants too (~0.4 GB saved)
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token  # LLaMA has no pad token by default
tokenizer.padding_side = "right"           # pad on right for causal LM training

print(f"Model loaded. Parameters: {model.num_parameters()/1e9:.1f}B")

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

Model loaded. Parameters: 8.0B


## 6. Attach LoRA Adapters

In [4]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Prepare the quantized model for training
# (enables gradient checkpointing, casts layer norms to float32)
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=LORA_MODULES,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# Expected: ~0.5–1% of total params — that's the LoRA adapter size

trainable params: 13,631,488 || all params: 8,043,892,736 || trainable%: 0.1695


GETTING WEIGHTS FROM HUGGING FACE

In [ ]:
from huggingface_hub import snapshot_download
import os

print("Downloading the latest checkpoint from Hugging Face...")

# This downloads your repository to Kaggle
local_dir = snapshot_download(
    repo_id="Saksham-Bali/llama-3.1-8b-legal-risk-qlora",
    local_dir="/kaggle/working/hf_checkpoints"
)

# This automatically targets the 'last-checkpoint' folder shown in your screenshot
checkpoint_path = os.path.join(local_dir, "last-checkpoint")
print(f"✓ Checkpoint ready to resume from: {checkpoint_path}")

## 7. Train

In [ ]:
from transformers import TrainingArguments, EarlyStoppingCallback
from trl import SFTTrainer

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    warmup_ratio=WARMUP_RATIO,
    lr_scheduler_type="cosine",
    bf16=True,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    report_to="none",   # set to "wandb" if you want experiment tracking
    seed=SEED,
    
    # ─── NEW: HUGGING FACE HUB SETTINGS ───────────────────────────
    push_to_hub=True,
    # This automatically creates a repo under your username
    hub_model_id="Saksham-Bali/llama-3.1-8b-legal-risk-qlora", 
    # Pushes the weights every time a checkpoint is saved (every 50 steps)
    hub_strategy="checkpoint",
    # Keeps your fine-tuned model private so no one else can see your weights
    hub_private_repo=True, 
    # ──────────────────────────────────────────────────────────────
)


trainer = SFTTrainer(
    model=model,
    args=training_args,          # Keep your original arguments!
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],  
    tokenizer=tokenizer,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LEN,
    callbacks = [EarlyStoppingCallback(early_stopping_patience = 3)]
)

# COMMENTED FOR NOW TO NOT TRAIN FROM SCRATCH
print("Starting training with Hub syncing...")
trainer.train()
print("Training complete.")

2026-04-27 20:24:58.355483: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777321498.553038     139 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777321498.607872     139 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777321499.066694     139 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777321499.066728     139 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777321499.066731     139 computation_placer.cc:177] computation placer alr

Map:   0%|          | 0/3975 [00:00<?, ? examples/s]

Map:   0%|          | 0/826 [00:00<?, ? examples/s]

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Starting training with Hub syncing...


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: The AccumulateGrad node's stream does not match the stream of the node that produced the incoming gradient. This may incur unnecessary synchronization and break CUDA graph capture if the AccumulateGrad node's stream is the default stream. This mismatch is caused by an AccumulateGrad node created prior to the current iteration being kept alive. This can happen if the autograd graph is still being kept alive by tensors such a

Step,Training Loss,Validation Loss
100,0.726300,0.718680
200,0.672000,0.683797


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


In [ ]:
import torch
from unittest.mock import patch

# 1. Save the original torch.load function
original_load = torch.load

# 2. Create a temporary wrapper that forces weights_only=False
def legacy_load(*args, **kwargs):
    kwargs['weights_only'] = False
    return original_load(*args, **kwargs)

print(f"Resuming training from {checkpoint_path}...")

# 3. Use the patch context manager to bypass the PyTorch warning
with patch('torch.load', side_effect=legacy_load):
    # This explicitly tells the trainer to pick up where it left off!
    trainer.train(resume_from_checkpoint=checkpoint_path)

print("Training complete.")

## 8. Save LoRA Adapter Weights
Weights are saved to `/kaggle/working/qlora_output/`. Download them from the **Output** tab on the right after the notebook finishes.

In [ ]:
import os, json as _json, datetime

# ── Save adapter + tokenizer ───────────────────────────────────
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

# ── Write a small run-info file so you remember what this checkpoint is ──
run_info = {
    "model_id": MODEL_ID,
    "lora_rank": LORA_RANK,
    "lora_alpha": LORA_ALPHA,
    "epochs": EPOCHS,
    "lr": LR,
    "batch_size": BATCH_SIZE,
    "grad_accum": GRAD_ACCUM,
    "max_seq_len": MAX_SEQ_LEN,
    "saved_at": datetime.datetime.now().isoformat(),
}
with open(os.path.join(OUTPUT_DIR, "run_info.json"), "w") as f:
    _json.dump(run_info, f, indent=2)

# ── Confirm files saved ───────────────────────────────────────
saved_files = os.listdir(OUTPUT_DIR)
print(f"\n✓ Adapter saved to: {OUTPUT_DIR}")
print(f"  Files: {sorted(saved_files)}")
print("\nTo reload later:")
print(f"  from peft import PeftModel")
print(f"  model = PeftModel.from_pretrained(base_model, '{OUTPUT_DIR}')")

ONLY FOR INFERENCE

In [ ]:
from peft import PeftModel

print(f"Attaching trained LoRA adapters from {checkpoint_path}...")
# This merges your Hugging Face weights with the 4-bit LLaMA model
model = PeftModel.from_pretrained(model, checkpoint_path)
print("✓ Fine-tuned weights attached successfully! Ready for inference.")

## 9. Evaluate — Extract Predictions on Val Set

We run inference on each val example, parse the predicted risk label out of the generated text, and compare against the ground truth.

In [ ]:
import json
import re
import torch
from tqdm import tqdm
from huggingface_hub import HfApi # <-- NEW: Hugging Face API

def extract_risk_label(generated_text: str) -> str:
    match = re.search(r"Risk Level:\s*(Low|Medium|High)", generated_text, re.IGNORECASE)
    if match: return match.group(1).capitalize()
    match = re.search(r"\b(Low|Medium|High)\b", generated_text, re.IGNORECASE)
    if match: return match.group(1).capitalize()
    return "Unknown"

def get_prompt_only(full_text: str) -> str:
    marker = "<|start_header_id|>assistant<|end_header_id|>\n\n"
    idx = full_text.find(marker)
    if idx != -1: return full_text[:idx + len(marker)]
    return full_text

def get_ground_truth_label(full_text: str) -> str:
    marker = "<|start_header_id|>assistant<|end_header_id|>\n\n"
    idx = full_text.find(marker)
    if idx != -1:
        response = full_text[idx + len(marker):]
        return extract_risk_label(response)
    return "Unknown"

# Load val examples
val_examples = []
with open(VAL_FILE) as f:
    for line in f:
        val_examples.append(json.loads(line.strip()))

print(f"Running inference on ALL {len(val_examples)} validation examples...")

model.eval()
y_true, y_pred, parties = [], [], []

# ─── NEW: CLOUD BACKUP SETUP ────────────────────────────────────
api = HfApi()
repo_id = "Saksham-Bali/llama-3.1-8b-legal-risk-qlora"
local_backup_path = "/kaggle/working/inference_backup.jsonl"

# Open the file in 'write' mode to create a fresh backup file
output_file = open(local_backup_path, "w") 
# ──────────────────────────────────────────────────────────────

for i, ex in enumerate(tqdm(val_examples)):
    full_text = ex["text"]
    prompt    = get_prompt_only(full_text)
    gt_label  = get_ground_truth_label(full_text)
    party     = ex.get("_meta", {}).get("party", "Unknown")

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_SEQ_LEN)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=80,
            do_sample=False,      
            pad_token_id=tokenizer.eos_token_id,
        )

    new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
    generated  = tokenizer.decode(new_tokens, skip_special_tokens=True)
    pred_label = extract_risk_label(generated)

    # ─── NEW: WRITE TO LOCAL FILE IMMEDIATELY ───────────────────
    result_dict = {
        "index": i,
        "true": gt_label, 
        "pred": pred_label, 
        "party": party
    }
    output_file.write(json.dumps(result_dict) + "\n")
    output_file.flush() # Forces it to write to the hard drive immediately
    # ──────────────────────────────────────────────────────────────

    y_true.append(gt_label)
    y_pred.append(pred_label)
    parties.append(party)

    # ─── NEW: UPLOAD TO HUGGING FACE EVERY 100 STEPS ──────────────
    if (i + 1) % 100 == 0:
        try:
            api.upload_file(
                path_or_fileobj=local_backup_path,
                path_in_repo="inference_backup.jsonl",
                repo_id=repo_id,
                repo_type="model"
            )
        except Exception as e:
            pass # If there is a tiny network blip, ignore it and keep generating
    # ──────────────────────────────────────────────────────────────

output_file.close()

# Final upload at the very end to ensure the last few examples are saved
api.upload_file(
    path_or_fileobj=local_backup_path,
    path_in_repo="inference_backup.jsonl",
    repo_id=repo_id,
    repo_type="model"
)
print("Inference complete and fully backed up to Hugging Face!")

## 10. Performance Metrics

In [ ]:
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
)
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

LABELS = ["Low", "Medium", "High"]

# ── Overall metrics ───────────────────────────────────────────
print("=" * 55)
print("OVERALL PERFORMANCE (val set)")
print("=" * 55)
print(f"Accuracy: {accuracy_score(y_true, y_pred):.3f}")
print()
print(classification_report(y_true, y_pred, labels=LABELS, zero_division=0))

# ── Per-party metrics ─────────────────────────────────────────
for party in ["Licensor", "Licensee"]:
    idx = [i for i, p in enumerate(parties) if p == party]
    if not idx:
        continue
    pt = [y_true[i] for i in idx]
    pp = [y_pred[i] for i in idx]
    print(f"{'=' * 55}")
    print(f"{party.upper()} (n={len(idx)})")
    print(f"{'=' * 55}")
    print(f"Accuracy: {accuracy_score(pt, pp):.3f}")
    print(classification_report(pt, pp, labels=LABELS, zero_division=0))

In [ ]:
# ── Confusion matrices ────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

splits = [
    ("Overall",  list(range(len(y_true))), y_true, y_pred),
    ("Licensor", [i for i,p in enumerate(parties) if p=="Licensor"], None, None),
    ("Licensee", [i for i,p in enumerate(parties) if p=="Licensee"], None, None),
]

for ax, (title, idx, yt, yp) in zip(axes, splits):
    if yt is None:
        yt = [y_true[i] for i in idx]
        yp = [y_pred[i] for i in idx]
    cm = confusion_matrix(yt, yp, labels=LABELS)
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True).clip(min=1)
    sns.heatmap(
        cm_norm, annot=True, fmt=".2f", cmap="Blues",
        xticklabels=LABELS, yticklabels=LABELS, ax=ax,
        vmin=0, vmax=1,
    )
    for i in range(len(LABELS)):
        for j in range(len(LABELS)):
            ax.text(j+0.5, i+0.72, f"({cm[i,j]})",
                    ha="center", va="center", fontsize=8, color="gray")
    ax.set_title(f"{title} (n={len(yt)})", fontsize=13)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")

plt.suptitle("Confusion Matrices — Fine-tuned LLaMA 3.1 8B (QLoRA)", fontsize=14, y=1.02)
plt.tight_layout()

cm_path = os.path.join(OUTPUT_DIR, "confusion_matrices.png")
plt.savefig(cm_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {cm_path}")

In [ ]:
# ── High↔Low flip analysis ────────────────────────────────────
flips = [
    (i, y_true[i], y_pred[i], parties[i])
    for i in range(len(y_true))
    if set([y_true[i], y_pred[i]]) == {"High", "Low"}
]

print(f"High↔Low flips: {len(flips)}")
if flips:
    print(f"{'Idx':>4}  {'True':>7}  {'Pred':>7}  Party")
    print("-" * 35)
    for idx, true, pred, party in flips:
        print(f"{idx:>4}  {true:>7}  {pred:>7}  {party}")

In [ ]:
# ── Prediction distribution ───────────────────────────────────
from collections import Counter

print("Prediction distribution (val):")
pred_dist = Counter(y_pred)
true_dist = Counter(y_true)
print(f"{'Label':>8}  {'True':>6}  {'Pred':>6}")
print("-" * 26)
for label in LABELS:
    print(f"{label:>8}  {true_dist[label]:>6}  {pred_dist[label]:>6}")

medium_frac = pred_dist.get("Medium", 0) / len(y_pred)
print(f"\nMedium prediction rate: {medium_frac:.1%}")
if medium_frac > 0.6:
    print("⚠️  Model may still be collapsing toward Medium — consider more epochs or checking annotations.")
else:
    print("✓  No medium-collapse detected.")

## 11. Qualitative Check — Sample Predictions

In [ ]:
import textwrap

def show_sample(idx):
    ex = val_examples[idx]
    prompt = get_prompt_only(ex["text"])
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_SEQ_LEN)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    with torch.no_grad():
        output_ids = model.generate(
            **inputs, max_new_tokens=120, do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
    generated = tokenizer.decode(new_tokens, skip_special_tokens=True)

    party  = ex.get("_meta", {}).get("party", "?")
    true_l = y_true[idx]
    pred_l = y_pred[idx]
    match  = "✓" if true_l == pred_l else "✗"

    print(f"{'─'*60}")
    print(f"Example #{idx}  |  Party: {party}  |  True: {true_l}  |  Pred: {pred_l}  {match}")
    print(f"{'─'*60}")
    print("Generated output:")
    print(textwrap.fill(generated.strip(), width=80))
    print()

correct_idx   = next((i for i in range(len(y_true)) if y_true[i] == y_pred[i]), 0)
incorrect_idx = next((i for i in range(len(y_true)) if y_true[i] != y_pred[i]), 1)
high_idx      = next((i for i in range(len(y_true)) if y_true[i] == "High"), 2)

for idx in [correct_idx, incorrect_idx, high_idx]:
    show_sample(idx)